In [ ]:
import pandas as pd # noqa: E402, F401
import json # noqa: E402, F401
from datetime import timedelta, datetime as dt # noqa: E402, F401
from dateutil.relativedelta import relativedelta # noqa: E402, F401
import os

from app.utilities.db_service import IbmDb2Creds, IbmDb2Service # noqa: E402, F401
from app.utilities.db_connections import live_connection # noqa: E402, F401

In [ ]:
conn = IbmDb2Service(live_connection)

In [ ]:
hb_opened_df = conn.df_query_sql_file('queries/home_branch_mems_opened.sql', begin_date='2026-04-01', end_date='2026-04-30')
hb_mems_df = conn.df_query_sql_file('queries/home_branch_mem_zips.sql', date='2026-04-30')
person_contact_df = conn.df_query_sql_file('queries/person_contact_info.sql')

# print(f"Homebranch opened len: {len(hb_opened_df)}")
# print(f"Homebranch total len: {len(hb_mems_df)}")

In [ ]:
hb_opened_df = conn.df_query_sql_file('queries/home_branch_mems_opened.sql', begin_date='2026-04-01', end_date='2026-04-30')
len(hb_opened_df)

In [ ]:
south_branch_zips = [
    64012, 64019, 64030, 64034, 64078, 64063, 64131, 64132, 64134, 64137, 64138, 64139,
    64145, 64146, 64147, 64149, 64170, 64197, 64701, 64720, 64724, 64725, 64726, 64730,
    64733, 64735, 64739, 64740, 64742, 64743, 64746, 64747, 64761, 64763, 64765, 64770,
    64779, 64780, 64783, 64788, 65323, 65338, 65360
]

overland_park_branch_zips = [
    66013, 66051, 66061, 66062, 66063, 66083, 66085, 66201, 66202, 66203, 66204,
    66206, 66207, 66208, 66209, 66210, 66211, 66212, 66213, 66214, 66215, 66216, 66217,
    66218, 66219, 66220, 66221, 66223, 66224, 66225, 66226, 66227, 66250, 66251, 66282,
    66285, 66266
]

north_branch_zips = [
    64144, 64120, 64188, 64162, 64117, 64161, 64168, 64150, 64119, 64118, 64151, 64158,
    64069, 64156, 64157, 64155, 64154, 64068, 64166, 64165, 64072, 64167, 64092, 64164,
    64153, 64195, 64190, 64163, 64073, 64028, 64089, 64060, 64077, 64024, 64079, 64492,
    64018, 64444, 64017, 64062, 64098, 64439, 64086, 64465, 64085, 64454, 64477, 64448,
    64671, 64493, 64440, 64401, 64035, 64484, 64490, 64507, 64650, 64504, 64443, 64637,
    64503, 64620, 64429, 64474, 64501, 64502, 64508, 64506, 64430, 64668, 64649, 64644,
    64505, 64625, 64436, 64422, 64689, 64459, 64497, 64469, 64656, 64633, 64421, 64485,
    64625, 64494, 64620, 64622, 64664, 64640, 64638, 64654, 64682, 64686, 64480, 64463,
    64473, 64449, 64670, 64639, 64623, 64427, 64647, 64601, 64643, 64657, 64451, 64457,
    64648, 64636, 64438, 64423, 64688, 64433
]

main_branch_zips = [
    64101, 64102, 64105, 64106, 64107, 64108, 64109, 64110, 64111, 64112, 64113, 64114,
    64116, 64121, 64123, 64124, 64125, 64126, 64127, 64128, 64129, 64130, 64141, 64148,
    64171, 64179, 64180, 64184, 64187, 64191, 64196, 64198, 64899, 64999
]

east_branch_zips = [
    64001, 64002, 64011, 64013, 64014, 64015, 64016, 64020, 64021, 64022, 64029, 64036,
    64037, 64040, 64050, 64051, 64052, 64053, 64054, 64055, 64056, 64057, 64058, 64061,
    64063, 64064, 64065, 64066, 64067, 64070, 64071, 64074, 64075, 64076, 64080, 64081,
    64082, 64086, 64088, 64090, 64093, 64096, 64097, 64136, 65236, 65286, 65301, 65302,
    65305, 65321, 65327, 65332, 65333, 65334, 65335, 65336, 65337, 65338, 65339, 65340,
    65344, 65351
]

# Confirm creation
# {
#     "South Branch Count": len(south_branch_zips),
#     "Overland Park Branch Count": len(overland_park_branch_zips),
#     "North Branch Count": len(north_branch_zips),
#     "Main Branch Count": len(main_branch_zips),
#     "East Branch Count": len(east_branch_zips)
# }

In [ ]:
# Adjusted function to account for ZIP and fallback to HOME_BRANCH
def assign_branch_with_fallback(row):
    zip_code = row["ZIP_cleaned"]
    home_branch = row["HOME_BRANCH"]

    if not pd.isna(zip_code):
        zip_code = int(zip_code)
        if zip_code in south_branch_zips:
            return "South Branch"
        elif zip_code in overland_park_branch_zips:
            return "Overland Park Branch"
        elif zip_code in north_branch_zips:
            return "North Branch"
        elif zip_code in east_branch_zips:
            return "East Branch"
        elif zip_code in main_branch_zips:
            return "Main Branch"

    # If ZIP doesn't match or is NaN, fallback to HOME_BRANCH if available
    if pd.notna(home_branch):
        return home_branch

    # Default fallback
    return "Unassigned"

# Apply cleaning and updated logic
hb_opened_df["ZIP_cleaned"] = pd.to_numeric(hb_opened_df["ZIP"], errors="coerce")
hb_opened_df["assigned_branch"] = hb_opened_df.apply(assign_branch_with_fallback, axis=1)


In [ ]:
## Run if you need to backfill

# from calendar import monthrange

# results = []

# start_year, start_month = 2025, 1
# end_year, end_month = dt.today().year, dt.today().month

# year, month = start_year, start_month
# while (year, month) <= (end_year, end_month):
#     begin_date = f"{year}-{month:02d}-01"
#     last_day = monthrange(year, month)[1]
#     end_date = f"{year}-{month:02d}-{last_day}"

#     df = conn.df_query_sql_file('queries/home_branch_mems_opened.sql', begin_date=begin_date, end_date=end_date)
#     df["ZIP_cleaned"] = pd.to_numeric(df["ZIP"], errors="coerce")
#     df["assigned_branch"] = df.apply(assign_branch_with_fallback, axis=1)

#     counts = df.groupby("assigned_branch").size().reset_index(name="Count")
#     counts.insert(0, "Month", f"{year}-{month:02d}")
#     results.append(counts)

#     if month == 12:
#         year += 1
#         month = 1
#     else:
#         month += 1

# all_months_df = pd.concat(results, ignore_index=True)
# print(results[0])
# print(results[-1])
# all_months_df.to_csv('monthly_mem_add_fix.csv')

In [ ]:
hb_mems_df["ZIP_cleaned"] = pd.to_numeric(hb_mems_df["ZIP"], errors="coerce")
hb_mems_df["assigned_branch"] = hb_mems_df.apply(assign_branch_with_fallback, axis=1)

In [ ]:
hb_mems_df.head()

In [ ]:
person_contact_df.head()

In [ ]:
hb_opened_df.groupby("assigned_branch").size().reset_index(name="Count")

In [ ]:
hb_mems_df.groupby("assigned_branch").size().reset_index(name="Count")